In [0]:
# Step 1: Build the ML Feature Table & Handle "Censored Demand"

In [0]:
# ── Cell 1: Imports ───────────────────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

In [0]:
fact_df   = spark.read.table("mini_project_team_a3t7.gold.fact_inventory_full")
demand_df = spark.read.table("mini_project_team_a3t7.gold.demand_intelligence1")
kpi_df    = spark.read.table("mini_project_team_a3t7.gold.inventory_kpis1")


In [0]:
kpi_df.columns

In [0]:
demand_df.columns

In [0]:
fact_df.columns

In [0]:
ml_base = (
    fact_df.alias("f")
    .join(
        demand_df.alias("d"),
        ["store_id", "product_id", "snapshot_date"],   # was inventory_date
        "left"
    )
    .join(
        kpi_df.alias("k"),
        ["store_id", "product_id", "snapshot_date"],   # was inventory_date
        "left"
    )

    # Impute true demand — fix stockout_flag (int 0/1) replaces is_stockout (bool)
    .withColumn("true_demand",
        F.when(
            F.col("k.stockout_flag") == 1,             # was k.is_stockout == True
            F.col("d.avg_units_7d")
        )
        .otherwise(F.col("f.units_sold"))
    )
)

In [0]:
w_future = (
    Window.partitionBy("store_id", "product_id")
          .orderBy("snapshot_date")                    # was inventory_date
          .rowsBetween(1, 14)
)

ml_features_full = (
    ml_base
    .withColumn("target_demand_14d",
                F.sum("true_demand").over(w_future))
    .select(
        # Keys
        "store_id",
        "product_id",
        "snapshot_date",                               # was inventory_date

        # supplier_id is already in fact_inventory_full — no dim_product join needed
        F.col("f.supplier_id").alias("supplier_id"),

        # current_stock_qty replaces current_stock (correct column name)
        F.col("f.current_stock_qty").alias("current_stock_qty"),

        # Demand features from demand_intelligence1
        F.col("d.seasonality_index"),
        F.col("d.trend_direction"),
        F.col("d.avg_units_30d"),

        # Category features from fact_inventory_full directly
        # price_tier does not exist — using gross_margin_pct as proxy for product value tier
        F.col("f.category"),
        F.col("f.brand"),

        # NOTE: store_size_tier removed — dim_store table not available
        # NOTE: price_tier removed — not a column in any available table

        # Target label
        "target_demand_14d"
    )
    # Fill nulls in numeric features to prevent ML pipeline errors
    .fillna({
        "seasonality_index": 1.0,
        "avg_units_30d":     0.0,
        "current_stock_qty": 0
    })
)

# Training set: rows where we have a valid 14-day lookahead label
training_data = ml_features_full.filter(
    F.col("target_demand_14d").isNotNull()
)

# Latest snapshot per store+product: used for generating today's predictions
w_latest = (
    Window.partitionBy("store_id", "product_id")
          .orderBy(F.col("snapshot_date").desc())     # was inventory_date
)
latest_features = (
    ml_features_full
    .withColumn("rn", F.row_number().over(w_latest))
    .filter(F.col("rn") == 1)
    .drop("rn")
)



In [0]:
print("Building and training Machine Learning model...")

# Categorical columns available after removing price_tier and store_size_tier
# Both were from dim_product / dim_store which don't exist
categorical_cols = [
    "trend_direction",   # from demand_intelligence1
    "category",          # from fact_inventory_full
    "brand",             # from fact_inventory_full — replaces price_tier
]

indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep")
    for c in categorical_cols
]

# Numeric feature columns
numeric_cols  = ["seasonality_index", "avg_units_30d"]
feature_cols  = numeric_cols + [c + "_idx" for c in categorical_cols]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep"
)

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="target_demand_14d",
    maxIter=20,
    maxDepth=5
)

ml_pipeline = Pipeline(stages=indexers + [assembler, gbt])
model       = ml_pipeline.fit(training_data)

print("Training Complete.")


In [0]:
print("Evaluating Model Accuracy...")

train_df, test_df = training_data.randomSplit([0.8, 0.2], seed=42)

eval_model       = ml_pipeline.fit(train_df)
test_predictions = eval_model.transform(test_df)

# Clamp negative predictions to zero — demand can't be negative
test_predictions = test_predictions.withColumn(
    "prediction",
    F.when(F.col("prediction") < 0, 0).otherwise(F.col("prediction"))
)

rmse_evaluator = RegressionEvaluator(
    labelCol="target_demand_14d", predictionCol="prediction", metricName="rmse")
mae_evaluator  = RegressionEvaluator(
    labelCol="target_demand_14d", predictionCol="prediction", metricName="mae")
r2_evaluator   = RegressionEvaluator(
    labelCol="target_demand_14d", predictionCol="prediction", metricName="r2")

rmse = rmse_evaluator.evaluate(test_predictions)
mae  = mae_evaluator.evaluate(test_predictions)
r2   = r2_evaluator.evaluate(test_predictions)

print("-" * 40)
print(f"Mean Absolute Error  (MAE) : {mae:.2f} units")
print(f"Root Mean Sq Error   (RMSE): {rmse:.2f} units")
print(f"R-Squared            (R2)  : {r2:.4f}")
print("-" * 40)


In [0]:
print("Generating predictions...")

preds_df = (
    model.transform(latest_features)
    .withColumnRenamed("prediction", "predicted_demand_14d")
    .withColumn(
        "predicted_demand_14d",
        F.when(F.col("predicted_demand_14d") < 0, 0)
         .otherwise(F.col("predicted_demand_14d"))
    )
)
display(preds_df)

In [0]:
# ── Cell 8: Reorder Logic & Safety Stock Calculation ─────────────────────────
# supplier_performance_fact does not exist — using fixed lead_time_variance = 2
# supplier_id is still carried through for reference in the output table

stock_recommendation = (
    preds_df.alias("pred")

    .withColumn("daily_run_rate",
        F.col("pred.predicted_demand_14d") / 14
    )

    # Safety stock = 10% buffer on forecast + 2-day lead time variance cover
    # lead_time_variance hardcoded to 2 (was from supplier_performance_fact)
    .withColumn("safety_stock",
        F.round(
            (F.col("pred.predicted_demand_14d") * 0.10) +
            (F.lit(2) * F.col("daily_run_rate"))          # lit(2) replaces sup.lead_time_variance
        )
    )

    .withColumn("optimal_stock_level",
        F.col("pred.predicted_demand_14d") + F.col("safety_stock")
    )

    # Suggested reorder = how much to order to reach optimal level
    # current_stock_qty replaces current_stock
    .withColumn("suggested_reorder_qty",
        F.greatest(
            F.lit(0),
            F.col("optimal_stock_level") -
            F.coalesce(F.col("pred.current_stock_qty"), F.lit(0))  # was current_stock
        )
    )

    .select(
        "store_id",
        "product_id",
        "supplier_id",
        F.round("pred.predicted_demand_14d", 0).alias("forecasted_14d_demand"),
        "safety_stock",
        # current_stock_qty is the correct column name in fact_inventory_full
        F.col("pred.current_stock_qty").alias("current_on_hand_stock"),  # was current_stock
        F.round("optimal_stock_level",   0).alias("target_stock_level"),
        F.round("suggested_reorder_qty", 0).alias("suggested_reorder_qty"),
        F.current_timestamp().alias("forecast_generated_at")
    )
)

In [0]:
display(spark.read.table("mini_project_team_a3t7.gold.inventory_reorder_recommendations"))

In [0]:
# ── Cell 9: Write to Gold Final Table ─────────────────────────────────────────
(
    stock_recommendation.write
    .format("delta")
    .mode("append")
    .saveAsTable("mini_project_team_a3t7.gold.inventory_reorder_recommendations")
)

print("Execution Complete! Saved to gold.inventory_reorder_recommendations")
display(
    spark.table("mini_project_team_a3t7.gold.inventory_reorder_recommendations")
    .limit(10)
)

In [0]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  Demand Forecast & Inventory Reorder Recommendations                    ║
# ║  Updated to use:                                                         ║
# ║    gold.fact_inventory_full   (was daily_inventory_fact)                 ║
# ║    gold.demand_intelligence1  (unchanged table name)                     ║
# ║    gold.inventory_kpis1       (unchanged table name)                     ║
# ║                                                                          ║
# ║  Key column fixes across all cells:                                      ║
# ║    inventory_date   → snapshot_date                                      ║
# ║    current_stock    → current_stock_qty                                  ║
# ║    is_stockout      → stockout_flag == 1                                 ║
# ║    price_tier       → removed (not in fact_inventory_full)               ║
# ║    store_size_tier  → removed (no dim_store available)                   ║
# ║    dim_product      → removed (product cols already in fact table)       ║
# ║    dim_store        → removed (not available)                            ║
# ║    supplier_performance_fact → removed (not available)                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝


# ── Cell 1: Imports ───────────────────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator


# ── Cell 2: Load Gold Tables ──────────────────────────────────────────────────
# THREE tables only — dim_product, dim_store, supplier_performance_fact
# do not exist; all needed columns are already in fact_inventory_full

fact_df   = spark.read.table("mini_project_team_a3t7.gold.fact_inventory_full")
demand_df = spark.read.table("mini_project_team_a3t7.gold.demand_intelligence1")
kpi_df    = spark.read.table("mini_project_team_a3t7.gold.inventory_kpis1")

# Quick schema check — uncomment to verify column names match
# fact_df.printSchema()
# demand_df.printSchema()
# kpi_df.printSchema()


# ── Cell 3: Build ML Base — Censored Demand Imputation ───────────────────────
# Join all three tables on the three-column grain key
# NOTE: join key is snapshot_date (was inventory_date in original)
#
# Censored demand logic:
#   When stockout_flag == 1, actual units_sold is artificially low
#   (couldn't sell because shelves were empty, not because demand was zero)
#   → substitute avg_units_7d from demand_intelligence as the true demand
#   Otherwise use actual units_sold from fact table

ml_base = (
    fact_df.alias("f")
    .join(
        demand_df.alias("d"),
        ["store_id", "product_id", "snapshot_date"],   # was inventory_date
        "left"
    )
    .join(
        kpi_df.alias("k"),
        ["store_id", "product_id", "snapshot_date"],   # was inventory_date
        "left"
    )

    # Impute true demand — fix stockout_flag (int 0/1) replaces is_stockout (bool)
    .withColumn("true_demand",
        F.when(
            F.col("k.stockout_flag") == 1,             # was k.is_stockout == True
            F.col("d.avg_units_7d")
        )
        .otherwise(F.col("f.units_sold"))
    )
)


# ── Cell 4: Define Forecast Target (14-Day Lookahead) ────────────────────────
# Window looks 1–14 rows FORWARD to sum future demand
# This is the label the model learns to predict
# NOTE: orderBy snapshot_date (was inventory_date)

w_future = (
    Window.partitionBy("store_id", "product_id")
          .orderBy("snapshot_date")                    # was inventory_date
          .rowsBetween(1, 14)
)

ml_features_full = (
    ml_base
    .withColumn("target_demand_14d",
                F.sum("true_demand").over(w_future))
    .select(
        # Keys
        "store_id",
        "product_id",
        "snapshot_date",                               # was inventory_date

        # supplier_id is already in fact_inventory_full — no dim_product join needed
        F.col("f.supplier_id").alias("supplier_id"),

        # current_stock_qty replaces current_stock (correct column name)
        F.col("f.current_stock_qty").alias("current_stock_qty"),

        # Demand features from demand_intelligence1
        F.col("d.seasonality_index"),
        F.col("d.trend_direction"),
        F.col("d.avg_units_30d"),

        # Category features from fact_inventory_full directly
        # price_tier does not exist — using gross_margin_pct as proxy for product value tier
        F.col("f.category"),
        F.col("f.brand"),

        # NOTE: store_size_tier removed — dim_store table not available
        # NOTE: price_tier removed — not a column in any available table

        # Target label
        "target_demand_14d"
    )
    # Fill nulls in numeric features to prevent ML pipeline errors
    .fillna({
        "seasonality_index": 1.0,
        "avg_units_30d":     0.0,
        "current_stock_qty": 0
    })
)

# Training set: rows where we have a valid 14-day lookahead label
training_data = ml_features_full.filter(
    F.col("target_demand_14d").isNotNull()
)

# Latest snapshot per store+product: used for generating today's predictions
w_latest = (
    Window.partitionBy("store_id", "product_id")
          .orderBy(F.col("snapshot_date").desc())     # was inventory_date
)
latest_features = (
    ml_features_full
    .withColumn("rn", F.row_number().over(w_latest))
    .filter(F.col("rn") == 1)
    .drop("rn")
)


# ── Cell 5: Build & Train the PySpark ML Pipeline ────────────────────────────
print("Building and training Machine Learning model...")

# Categorical columns available after removing price_tier and store_size_tier
# Both were from dim_product / dim_store which don't exist
categorical_cols = [
    "trend_direction",   # from demand_intelligence1
    "category",          # from fact_inventory_full
    "brand",             # from fact_inventory_full — replaces price_tier
]

indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep")
    for c in categorical_cols
]

# Numeric feature columns
numeric_cols  = ["seasonality_index", "avg_units_30d"]
feature_cols  = numeric_cols + [c + "_idx" for c in categorical_cols]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep"
)

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="target_demand_14d",
    maxIter=20,
    maxDepth=5
)

ml_pipeline = Pipeline(stages=indexers + [assembler, gbt])
model       = ml_pipeline.fit(training_data)

print("Training Complete.")


# ── Cell 6: Evaluate Model Accuracy (80/20 split) ────────────────────────────
print("Evaluating Model Accuracy...")

train_df, test_df = training_data.randomSplit([0.8, 0.2], seed=42)

eval_model       = ml_pipeline.fit(train_df)
test_predictions = eval_model.transform(test_df)

# Clamp negative predictions to zero — demand can't be negative
test_predictions = test_predictions.withColumn(
    "prediction",
    F.when(F.col("prediction") < 0, 0).otherwise(F.col("prediction"))
)

rmse_evaluator = RegressionEvaluator(
    labelCol="target_demand_14d", predictionCol="prediction", metricName="rmse")
mae_evaluator  = RegressionEvaluator(
    labelCol="target_demand_14d", predictionCol="prediction", metricName="mae")
r2_evaluator   = RegressionEvaluator(
    labelCol="target_demand_14d", predictionCol="prediction", metricName="r2")

rmse = rmse_evaluator.evaluate(test_predictions)
mae  = mae_evaluator.evaluate(test_predictions)
r2   = r2_evaluator.evaluate(test_predictions)

print("-" * 40)
print(f"Mean Absolute Error  (MAE) : {mae:.2f} units")
print(f"Root Mean Sq Error   (RMSE): {rmse:.2f} units")
print(f"R-Squared            (R2)  : {r2:.4f}")
print("-" * 40)


# ── Cell 7: Generate Predictions for Latest Snapshot ─────────────────────────
print("Generating predictions...")

preds_df = (
    model.transform(latest_features)
    .withColumnRenamed("prediction", "predicted_demand_14d")
    .withColumn(
        "predicted_demand_14d",
        F.when(F.col("predicted_demand_14d") < 0, 0)
         .otherwise(F.col("predicted_demand_14d"))
    )
)


# ── Cell 8: Reorder Logic & Safety Stock Calculation ─────────────────────────
# supplier_performance_fact does not exist — using fixed lead_time_variance = 2
# supplier_id is still carried through for reference in the output table

stock_recommendation = (
    preds_df.alias("pred")

    .withColumn("daily_run_rate",
        F.col("pred.predicted_demand_14d") / 14
    )

    # Safety stock = 10% buffer on forecast + 2-day lead time variance cover
    # lead_time_variance hardcoded to 2 (was from supplier_performance_fact)
    .withColumn("safety_stock",
        F.round(
            (F.col("pred.predicted_demand_14d") * 0.10) +
            (F.lit(2) * F.col("daily_run_rate"))          # lit(2) replaces sup.lead_time_variance
        )
    )

    .withColumn("optimal_stock_level",
        F.col("pred.predicted_demand_14d") + F.col("safety_stock")
    )

    # Suggested reorder = how much to order to reach optimal level
    # current_stock_qty replaces current_stock
    .withColumn("suggested_reorder_qty",
        F.greatest(
            F.lit(0),
            F.col("optimal_stock_level") -
            F.coalesce(F.col("pred.current_stock_qty"), F.lit(0))  # was current_stock
        )
    )

    .select(
        "store_id",
        "product_id",
        "supplier_id",
        F.round("pred.predicted_demand_14d", 0).alias("forecasted_14d_demand"),
        "safety_stock",
        # current_stock_qty is the correct column name in fact_inventory_full
        F.col("pred.current_stock_qty").alias("current_on_hand_stock"),  # was current_stock
        F.round("optimal_stock_level",   0).alias("target_stock_level"),
        F.round("suggested_reorder_qty", 0).alias("suggested_reorder_qty"),
        F.current_timestamp().alias("forecast_generated_at")
    )
)


# ── Cell 9: Write to Gold Final Table ─────────────────────────────────────────
(
    stock_recommendation.write
    .format("delta")
    .mode("append")
    .saveAsTable("mini_project_team_a3t7.gold.inventory_reorder_recommendations")
)

# print("Execution Complete! Saved to gold.inventory_reorder_recommendations")
# display(
#     spark.table("mini_project_team_a3t7.gold.inventory_reorder_recommendations")
#     .limit(10)
# )

## 2nd algoritham

In [0]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  Demand Forecast v4 — Tuned GBT Only                                    ║
# ║  Random Forest removed — exceeded 1GB Spark Connect model size limit     ║
# ║  Linear Regression removed — keeping single best algorithm               ║
# ║                                                                          ║
# ║  GBT tuning applied:                                                     ║
# ║    maxIter=50, maxDepth=4, stepSize=0.05, subsamplingRate=0.8            ║
# ║  These settings keep model size small while improving over default GBT   ║
# ╚══════════════════════════════════════════════════════════════════════════╝

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# ── Step 1: Load Gold Tables ──────────────────────────────────────────────────
fact_df   = spark.table("mini_project_team_a3t7.gold.fact_inventory_full")
demand_df = spark.table("mini_project_team_a3t7.gold.demand_intelligence1")
kpi_df    = spark.table("mini_project_team_a3t7.gold.inventory_kpis1")

# ── Step 2: Join & resolve all ambiguous columns immediately ──────────────────
ml_base = (
    fact_df.alias("f")
    .join(
        demand_df.alias("d"),
        ["store_id", "product_id", "snapshot_date"],
        "left"
    )
    .join(
        kpi_df.alias("k"),
        ["store_id", "product_id", "snapshot_date"],
        "left"
    )
    # Resolve all ambiguous columns right after join
    .select(
        F.col("store_id"),
        F.col("product_id"),
        F.col("snapshot_date"),
        F.col("f.supplier_id"),
        F.col("f.current_stock_qty"),
        F.col("f.units_sold"),
        F.col("f.category"),
        F.col("f.brand"),
        F.col("f.gross_margin_pct"),
        F.col("f.sell_through_rate"),
        F.col("f.stock_cover_days"),
        F.col("f.reorder_flag"),
        F.col("f.stockout_flag"),
        F.col("f.max_stock"),
        F.col("f.available_stock_qty"),
        F.col("d.seasonality_index"),
        F.col("d.trend_direction"),
        F.col("d.avg_units_7d").alias("d_avg_units_7d"),
        F.col("d.avg_units_30d").alias("d_avg_units_30d"),
        F.col("k.stockout_flag").alias("k_stockout_flag"),
        F.col("k.inventory_health"),
    )
    # Censored demand imputation
    .withColumn("true_demand",
        F.when(F.col("k_stockout_flag") == 1, F.col("d_avg_units_7d"))
         .otherwise(F.col("units_sold"))
    )
)

# ── Step 3: Feature Engineering ───────────────────────────────────────────────
w_lag = Window.partitionBy("store_id", "product_id").orderBy("snapshot_date")
w7    = Window.partitionBy("store_id", "product_id").orderBy("snapshot_date").rowsBetween(-7,  0)
w14   = Window.partitionBy("store_id", "product_id").orderBy("snapshot_date").rowsBetween(-14, 0)
w30   = Window.partitionBy("store_id", "product_id").orderBy("snapshot_date").rowsBetween(-30, 0)
w_fwd = Window.partitionBy("store_id", "product_id").orderBy("snapshot_date").rowsBetween(1, 14)

ml_engineered = (
    ml_base

    # Lag features
    .withColumn("lag_units_1d",  F.lag("true_demand", 1).over(w_lag))
    .withColumn("lag_units_7d",  F.lag("true_demand", 7).over(w_lag))
    .withColumn("lag_units_14d", F.lag("true_demand", 14).over(w_lag))

    # Rolling demand averages
    .withColumn("avg_demand_7d",  F.round(F.avg("true_demand").over(w7),  2))
    .withColumn("avg_demand_14d", F.round(F.avg("true_demand").over(w14), 2))
    .withColumn("avg_demand_30d", F.round(F.avg("true_demand").over(w30), 2))

    # Demand volatility
    .withColumn("demand_stddev_14d",
        F.round(F.stddev("true_demand").over(w14), 3))

    # Demand acceleration: 7d avg / 30d avg
    .withColumn("demand_acceleration",
        F.round(
            F.when(F.col("avg_demand_30d") > 0,
                   F.col("avg_demand_7d") / F.col("avg_demand_30d"))
             .otherwise(F.lit(1.0)),
            3
        )
    )

    # Stock pressure: 1 / stock_cover_days
    .withColumn("stock_pressure",
        F.round(
            F.when(
                F.col("stock_cover_days").isNotNull() &
                (F.col("stock_cover_days") > 0),
                F.lit(1.0) / F.col("stock_cover_days")
            ).otherwise(F.lit(0.0)),
            4
        )
    )

    # Stock utilisation ratio
    .withColumn("stock_util_ratio",
        F.round(
            F.when(F.col("max_stock") > 0,
                   F.col("current_stock_qty") / F.col("max_stock"))
             .otherwise(F.lit(0.0)),
            3
        )
    )

    # Cleaned sell_through and margin
    .withColumn("sell_through_rate_clean",
        F.coalesce(F.col("sell_through_rate"), F.lit(0.0)))
    .withColumn("gross_margin_clean",
        F.coalesce(F.col("gross_margin_pct"), F.lit(0.0)))

    # Date features
    .withColumn("day_of_week", F.dayofweek("snapshot_date").cast("double"))
    .withColumn("month_num",   F.month("snapshot_date").cast("double"))
    .withColumn("is_weekend",
        F.when(F.dayofweek("snapshot_date").isin(1, 7), 1.0).otherwise(0.0))

    # Seasonality index
    .withColumn("seasonality_idx",
        F.coalesce(F.col("seasonality_index"), F.lit(1.0)))

    # Target label
    .withColumn("target_demand_14d", F.sum("true_demand").over(w_fwd))
)

# ── Step 4: Final feature selection ───────────────────────────────────────────
NUMERIC_FEATURES = [
    "lag_units_1d",
    "lag_units_7d",
    "lag_units_14d",
    "avg_demand_7d",
    "avg_demand_14d",
    "avg_demand_30d",
    "demand_stddev_14d",
    "demand_acceleration",
    "stock_pressure",
    "stock_util_ratio",
    "sell_through_rate_clean",
    "gross_margin_clean",
    "seasonality_idx",
    "day_of_week",
    "month_num",
    "is_weekend",
]

CATEGORICAL_FEATURES = [
    "trend_direction",
    "category",
    "brand",
]

ml_features_full = (
    ml_engineered
    .select(
        "store_id", "product_id", "snapshot_date",
        "supplier_id",
        "current_stock_qty",
        *NUMERIC_FEATURES,
        *CATEGORICAL_FEATURES,
        "target_demand_14d"
    )
    .fillna({
        "lag_units_1d":            0.0,
        "lag_units_7d":            0.0,
        "lag_units_14d":           0.0,
        "avg_demand_7d":           0.0,
        "avg_demand_14d":          0.0,
        "avg_demand_30d":          0.0,
        "demand_stddev_14d":       0.0,
        "demand_acceleration":     1.0,
        "stock_pressure":          0.0,
        "stock_util_ratio":        0.0,
        "sell_through_rate_clean": 0.0,
        "gross_margin_clean":      0.0,
        "seasonality_idx":         1.0,
        "current_stock_qty":       0,
    })
)

training_data = ml_features_full.filter(F.col("target_demand_14d").isNotNull())

w_latest = Window.partitionBy("store_id", "product_id").orderBy(F.col("snapshot_date").desc())
latest_features = (
    ml_features_full
    .withColumn("rn", F.row_number().over(w_latest))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# ── Step 5: Build ML Pipeline — Tuned GBT only ───────────────────────────────
# maxIter=50   → more trees than default (20) for better fit
# maxDepth=4   → shallower than default (5) keeps model small
# stepSize=0.05 → smaller learning rate for stable convergence
# subsamplingRate=0.8 → uses 80% of data per tree, reduces overfitting
# These four params together keep model well under the 1GB Spark Connect limit

indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep")
    for c in CATEGORICAL_FEATURES
]

feature_cols = NUMERIC_FEATURES + [c + "_idx" for c in CATEGORICAL_FEATURES]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="keep"
)

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="target_demand_14d",
    maxIter=50,            # 50 trees — balanced accuracy vs size
    maxDepth=4,            # shallow — key to staying under 1GB limit
    stepSize=0.05,         # small learning rate
    subsamplingRate=0.8,   # row sampling per tree
)

ml_pipeline = Pipeline(stages=indexers + [assembler, gbt])

# ── Step 6: Train & Evaluate ──────────────────────────────────────────────────
print("Training Tuned GBT model...")

train_df, test_df = training_data.randomSplit([0.8, 0.2], seed=42)

model            = ml_pipeline.fit(train_df)
test_predictions = model.transform(test_df)
test_predictions = test_predictions.withColumn(
    "prediction",
    F.when(F.col("prediction") < 0, 0).otherwise(F.col("prediction"))
)

mae_eval  = RegressionEvaluator(labelCol="target_demand_14d", predictionCol="prediction", metricName="mae")
rmse_eval = RegressionEvaluator(labelCol="target_demand_14d", predictionCol="prediction", metricName="rmse")
r2_eval   = RegressionEvaluator(labelCol="target_demand_14d", predictionCol="prediction", metricName="r2")

mae  = mae_eval.evaluate(test_predictions)
rmse = rmse_eval.evaluate(test_predictions)
r2   = r2_eval.evaluate(test_predictions)

print("-" * 40)
print(f"Algorithm            : Tuned GBT")
print(f"Mean Absolute Error  : {mae:.2f} units")
print(f"Root Mean Sq Error   : {rmse:.2f} units")
print(f"R-Squared            : {r2:.4f}")
print(f"Improvement over v1  : MAE {11.61 - mae:+.2f}  R² {r2 - 0.4984:+.4f}")
print("-" * 40)

# ── Step 7: Train final model on ALL training data ────────────────────────────
# Retrain on full dataset (not just 80%) for best prediction quality
print("\nRetraining on full dataset for final predictions...")
final_model = ml_pipeline.fit(training_data)

# ── Step 8: Generate Predictions ─────────────────────────────────────────────
print("Generating predictions...")
preds_df = (
    final_model.transform(latest_features)
    .withColumnRenamed("prediction", "predicted_demand_14d")
    .withColumn("predicted_demand_14d",
        F.when(F.col("predicted_demand_14d") < 0, 0)
         .otherwise(F.col("predicted_demand_14d")))
)

# ── Step 9: Reorder Recommendations ──────────────────────────────────────────
stock_recommendation = (
    preds_df.alias("pred")
    .withColumn("daily_run_rate",
        F.col("pred.predicted_demand_14d") / 14)
    .withColumn("safety_stock",
        F.round(
            (F.col("pred.predicted_demand_14d") * 0.10) +
            (F.lit(2) * F.col("daily_run_rate"))
        )
    )
    .withColumn("optimal_stock_level",
        F.col("pred.predicted_demand_14d") + F.col("safety_stock"))
    .withColumn("suggested_reorder_qty",
        F.greatest(
            F.lit(0),
            F.col("optimal_stock_level") -
            F.coalesce(F.col("pred.current_stock_qty"), F.lit(0))
        )
    )
    .withColumn("model_used", F.lit("GBT_tuned_v4"))
    .select(
        "store_id", "product_id", "supplier_id",
        F.round("pred.predicted_demand_14d", 0).alias("forecasted_14d_demand"),
        "safety_stock",
        F.col("pred.current_stock_qty").alias("current_on_hand_stock"),
        F.round("optimal_stock_level",   0).alias("target_stock_level"),
        F.round("suggested_reorder_qty", 0).alias("suggested_reorder_qty"),
        "model_used",
        F.current_timestamp().alias("forecast_generated_at")
    )
)

# ── Step 10: Write to Gold ────────────────────────────────────────────────────
(
    stock_recommendation.write
    .format("delta")
    .mode("append")
    .saveAsTable("mini_project_team_a3t7.gold.inventory_reorder_recommendations")
)

print("\nDone. Saved → gold.inventory_reorder_recommendations")
# display(
#     spark.table("mini_project_team_a3t7.gold.inventory_reorder_recommendations")
#     .limit(10)
# )

In [0]:
fact_df   = spark.table("mini_project_team_a3t7.gold.fact_inventory_full")
demand_df = spark.table("mini_project_team_a3t7.gold.demand_intelligence1")
kpi_df    = spark.table("mini_project_team_a3t7.gold.inventory_kpis1")

In [0]:
ml_features_full.columns

In [0]:
fact_df.display()

In [0]:
training_data.columns

In [0]:
spark.read.table()